In [1]:
# Cell 1 — B1: environment and hardware verification
import sys
import shutil
import subprocess
import importlib.metadata as metadata
import torch

print("Baseline: B1 — no pruning, 100% visual tokens")
print("Dataset: SLAKE")
print("Python:", sys.version.split()[0])

for package in [
    "torch", "torchvision", "transformers",
    "accelerate", "huggingface-hub", "numpy"
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

print("\nCUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

assert torch.cuda.is_available(), "Enable the GPU accelerator in Kaggle."
assert torch.cuda.device_count() == 2, "Select GPU T4 x2 in Kaggle settings."

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Total memory: {total / 2**30:.2f} GiB")
    print(f"  Free memory:  {free / 2**30:.2f} GiB")

disk = shutil.disk_usage("/kaggle/working")
print(f"\nFree disk: {disk.free / 2**30:.2f} GiB")

if shutil.which("nvidia-smi"):
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        check=False
    )
    print("\n" + result.stdout)

Baseline: B1 — no pruning, 100% visual tokens
Dataset: SLAKE
Python: 3.13.15
torch: 2.11.0+cu128
torchvision: 0.26.0+cu128
transformers: 5.16.1
accelerate: 1.14.0
huggingface-hub: 1.29.0
numpy: 2.1.3

CUDA version: 12.8
CUDA available: True
GPU count: 2

GPU 0: Tesla T4
  Total memory: 14.56 GiB
  Free memory:  14.46 GiB

GPU 1: Tesla T4
  Total memory: 14.56 GiB
  Free memory:  14.46 GiB

Free disk: 19.50 GiB

Wed Oct 14 21:27:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|

In [2]:
from pathlib import Path
import shutil
import subprocess
import sys

repo = Path("/kaggle/working/LLaVA-Med")

if not repo.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/microsoft/LLaVA-Med.git",
            str(repo),
        ],
        check=True,
    )
else:
    assert (repo / ".git").is_dir(), f"{repo} exists but is not a Git repository"

commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    text=True,
).strip()
Path("/kaggle/working/b1_llava_med_commit.txt").write_text(commit + "\n")
print("LLaVA-Med commit:", commit)

# Keep Kaggle's CUDA-enabled PyTorch. Install the repository's inference
# stack, with a Python-3.13-compatible SentencePiece release.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "transformers==4.36.2",
        "accelerate==0.21.0",
        "sentencepiece==0.2.2",
        "einops==0.6.1",
        "shortuuid",
    ],
    check=True,
)

# Install Microsoft's repository code without pulling in its separate
# training, serving, and evaluation dependencies.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "--no-deps", "-e", str(repo),
    ],
    check=True,
)

# Check imports in a fresh Python process, avoiding stale notebook imports.
probe = """
import torch
import transformers
import accelerate
import sentencepiece
from llava.model import LlavaMistralForCausalLM
from llava.conversation import conv_templates
from llava.mm_utils import tokenizer_image_token

print("IMPORT CHECK PASSED")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("SentencePiece:", sentencepiece.__version__)
print("CUDA GPUs:", torch.cuda.device_count())
print("Mistral conversation template:", "mistral_instruct" in conv_templates)
"""
subprocess.run([sys.executable, "-c", probe], check=True)

free_gib = shutil.disk_usage("/kaggle/working").free / 2**30
print(f"Free disk after setup: {free_gib:.2f} GiB")

Cloning into '/kaggle/working/LLaVA-Med'...


LLaVA-Med commit: 30697ca50b5c29a8e955c99330b259776aef27b9
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 236.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 347.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 265.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 375.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 249.8 MB/s eta 0:00:00
  Attempting uninstall: einops
    Found existing installation: einops 0.8.2
    Uninstalling einops-0.8.2:
      Successfully uninstalled einops-0.8.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.85.7 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.15.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.


Obtaining file:///kaggle/working/LLaVA-Med
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for llava_med (pyproject.toml): started
  Building editable for llava_med (pyproject.toml): finished with status 'done'
  Created wheel for llava_med: filename=llava_med-1.5.0-0.editable-py3-none-any.whl size=12972 sha256=484a4bb0408dcb8f3f527c025df9156306045e52bd343991971365bf13a68337
  Stored in directory: /tmp/pip-ephem-wheel-cache-bvn9ple0/wheels/55/30/60/bcae4c68bb449dcc77b765e793887c9f42cf47fe1ad37aa0cf
Successfully built llava_med

/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


IMPORT CHECK PASSED
PyTorch: 2.11.0+cu128
Transformers: 4.36.2
Accelerate: 0.21.0
SentencePiece: 0.2.2
CUDA GPUs: 2
Mistral conversation template: True
Free disk after setup: 19.50 GiB


In [3]:
from collections import Counter
from pathlib import Path, PurePosixPath
import json
import shutil
import zipfile

from huggingface_hub import HfApi, hf_hub_download
from PIL import Image

DATASET_ID = "BoKelvin/SLAKE"
SLAKE_DIR = Path("/kaggle/working/slake")
SLAKE_DIR.mkdir(parents=True, exist_ok=True)

# Pin the precise dataset revision used by this experiment.
dataset_sha = HfApi().repo_info(
    repo_id=DATASET_ID,
    repo_type="dataset",
).sha
(SLAKE_DIR / "dataset_revision.txt").write_text(dataset_sha + "\n")
print("SLAKE revision:", dataset_sha)

filenames = {
    "train": "train.json",
    "validation": "validation.json",
    "test": "test.json",
}

splits = {}
for split_name, filename in filenames.items():
    downloaded = hf_hub_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        revision=dataset_sha,
        filename=filename,
    )
    splits[split_name] = json.loads(Path(downloaded).read_text())
    assert isinstance(splits[split_name], list)

image_archive = hf_hub_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    revision=dataset_sha,
    filename="imgs.zip",
)

# Extract only the source images referenced by the three official splits.
image_names = {
    row["img_name"]
    for rows in splits.values()
    for row in rows
}
image_dir = SLAKE_DIR / "imgs"

with zipfile.ZipFile(image_archive) as archive:
    archive_names = archive.namelist()

    for image_name in sorted(image_names):
        relative = PurePosixPath(image_name)
        assert not relative.is_absolute() and ".." not in relative.parts

        matches = [
            name for name in archive_names
            if name == image_name or name.endswith("/" + image_name)
        ]
        assert len(matches) == 1, (
            f"Expected one archive match for {image_name}, got {matches}"
        )

        destination = image_dir.joinpath(*relative.parts)
        if not destination.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(matches[0]) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)

english_splits = {
    name: [row for row in rows if row["q_lang"] == "en"]
    for name, rows in splits.items()
}

for name, rows in splits.items():
    english = english_splits[name]
    print(f"\n{name}: {len(rows)} total questions; {len(english)} English questions")
    print("  English answer types:", dict(Counter(
        row["answer_type"] for row in english
    )))
    print("  English base types:", dict(Counter(
        row["base_type"] for row in english
    )))
    print("  Distinct English images:", len({
        row["img_name"] for row in english
    }))

split_images = {
    name: {row["img_name"] for row in rows}
    for name, rows in splits.items()
}
print("\nImage overlap between splits:")
for left, right in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    print(f"  {left} ∩ {right}: {len(split_images[left] & split_images[right])}")

assert english_splits["test"], "No English test questions found"
assert all(
    (image_dir / row["img_name"]).is_file()
    for rows in english_splits.values()
    for row in rows
), "One or more source images are missing"

sample = english_splits["train"][0]
with Image.open(image_dir / sample["img_name"]) as image:
    image.verify()

print("\nSample training question:", sample["question"])
print("Sample training answer:", sample["answer"])
print("Image check: passed")
print(f"Extracted image paths: {len(image_names)}")
print(f"Free disk: {shutil.disk_usage('/kaggle/working').free / 2**30:.2f} GiB")

SLAKE revision: a9083ce6c34ac3ffb17671a605962924d8a8f9e9


train.json: 0.00B [00:00, ?B/s]

validation.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

imgs.zip:   0%|          | 0.00/212M [00:00<?, ?B/s]


train: 9835 total questions; 4919 English questions
  English answer types: {'OPEN': 2976, 'CLOSED': 1943}
  English base types: {'vqa': 4337, 'kvqa': 582}
  Distinct English images: 450

validation: 2099 total questions; 1053 English questions
  English answer types: {'OPEN': 631, 'CLOSED': 422}
  English base types: {'vqa': 898, 'kvqa': 155}
  Distinct English images: 96

test: 2094 total questions; 1061 English questions
  English answer types: {'OPEN': 645, 'CLOSED': 416}
  English base types: {'vqa': 913, 'kvqa': 148}
  Distinct English images: 96

Image overlap between splits:
  train ∩ validation: 130
  train ∩ test: 142
  validation ∩ test: 26

Sample training question: What modality is used to take this image?
Sample training answer: MRI
Image check: passed
Extracted image paths: 642
Free disk: 19.44 GiB


In [4]:
from pathlib import Path
import json
import shutil

from huggingface_hub import HfApi, hf_hub_download, snapshot_download

# Verify image separation for the English-only experiment.
english_images = {
    name: {row["img_name"] for row in rows}
    for name, rows in english_splits.items()
}
for left, right in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    overlap = english_images[left] & english_images[right]
    print(f"English {left} ∩ {right}: {len(overlap)} images")
    assert not overlap, f"English image leakage between {left} and {right}"

MODEL_ID = "microsoft/llava-med-v1.5-mistral-7b"
MODEL_DIR = Path("/kaggle/working/b1_llava_med_model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Pin the model revision and inspect its index before the large download.
model_sha = HfApi().model_info(MODEL_ID).sha
index_path = hf_hub_download(
    repo_id=MODEL_ID,
    filename="model.safetensors.index.json",
    revision=model_sha,
    local_dir=str(MODEL_DIR),
)
index = json.loads(Path(index_path).read_text())
weight_bytes = index["metadata"]["total_size"]

free_bytes = shutil.disk_usage("/kaggle/working").free
reserve_bytes = int(2.5 * 2**30)

print("\nModel revision:", model_sha)
print(f"Checkpoint weights: {weight_bytes / 2**30:.2f} GiB")
print(f"Free disk now:      {free_bytes / 2**30:.2f} GiB")
assert free_bytes > weight_bytes + reserve_bytes, (
    "Insufficient disk space for the checkpoint and subsequent setup."
)

# local_dir stores the files in one place, avoiding a second full copy
# of the checkpoint in the standard Hub cache.
snapshot_download(
    repo_id=MODEL_ID,
    revision=model_sha,
    local_dir=str(MODEL_DIR),
    allow_patterns=[
        "*.json",
        "*.safetensors",
        "tokenizer.model",
    ],
    max_workers=1,
)

shards = sorted(set(index["weight_map"].values()))
missing = [name for name in shards if not (MODEL_DIR / name).is_file()]
assert not missing, f"Missing weight shards: {missing}"
assert (MODEL_DIR / "config.json").is_file()
assert (MODEL_DIR / "tokenizer.model").is_file()

Path("/kaggle/working/b1_model_revision.txt").write_text(
    model_sha + "\n"
)

print("\nDownload complete.")
print("Weight shards:", shards)
print(f"Free disk remaining: "
      f"{shutil.disk_usage('/kaggle/working').free / 2**30:.2f} GiB")

English train ∩ validation: 0 images
English train ∩ test: 0 images
English validation ∩ test: 0 images


model.safetensors.index.json: 0.00B [00:00, ?B/s]


Model revision: 91bb16c122001ddc9cf1fd36ce1dae09448943a2
Checkpoint weights: 14.09 GiB
Free disk now:      19.44 GiB


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/262M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]


Download complete.
Weight shards: ['model-00001-of-00004.safetensors', 'model-00002-of-00004.safetensors', 'model-00003-of-00004.safetensors', 'model-00004-of-00004.safetensors']
Free disk remaining: 5.35 GiB


In [6]:
# Cell 5a — expose the cloned Microsoft repository to this notebook kernel
from pathlib import Path
import sys

repo = Path("/kaggle/working/LLaVA-Med")
assert (repo / "llava" / "model" / "__init__.py").is_file(), (
    "The repository is missing; rerun Cell 2."
)

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from llava.model import LlavaMistralForCausalLM
print("LLaVA-Med import: passed")
print("Repository:", repo)

/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


LLaVA-Med import: passed
Repository: /kaggle/working/LLaVA-Med


/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [7]:
import torch
from transformers import AutoTokenizer
from llava.model import LlavaMistralForCausalLM

assert torch.cuda.device_count() == 2
model_path = str(MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)

# Place embeddings, vision tower, projector, and the first half of
# Mistral on GPU 0; place the remaining layers and output on GPU 1.
device_map = {
    "model.embed_tokens": 0,
    "model.vision_tower": 0,
    "model.mm_projector": 0,
    "model.norm": 1,
    "lm_head": 1,
}
device_map.update({
    f"model.layers.{layer}": 0 if layer < 16 else 1
    for layer in range(32)
})

torch.cuda.empty_cache()

model = LlavaMistralForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map=device_map,
    use_flash_attention_2=False,
)

assert model.config.num_hidden_layers == 32
assert model.config.mm_use_im_patch_token is False
assert model.config.mm_use_im_start_end is False
assert len(tokenizer) == model.get_input_embeddings().num_embeddings, (
    "Tokenizer and embedding sizes differ; stop before running inference."
)

# The official repository loads this CLIP tower after constructing the
# language model. Keep it on GPU 0 with the multimodal projector.
vision_tower = model.get_vision_tower()
if not vision_tower.is_loaded:
    vision_tower.load_model()
vision_tower.to(device="cuda:0", dtype=torch.float16)
image_processor = vision_tower.image_processor

model.requires_grad_(False)
model.eval()
torch.cuda.synchronize(0)
torch.cuda.synchronize(1)

print("Model:", MODEL_ID)
print("Precision: FP16")
print("Trainable parameters:", sum(p.numel() for p in model.parameters()
                                   if p.requires_grad))
print("Vision patches per image:", vision_tower.num_patches)
print("Embeddings:", model.get_input_embeddings().weight.device)
print("Mistral layer 0:", next(model.model.layers[0].parameters()).device)
print("Mistral layer 16:", next(model.model.layers[16].parameters()).device)
print("Vision tower:", next(vision_tower.parameters()).device)
print("Projector:", next(model.model.mm_projector.parameters()).device)
print("Output head:", model.lm_head.weight.device)

for gpu in range(2):
    allocated = torch.cuda.memory_allocated(gpu) / 2**30
    free, total = torch.cuda.mem_get_info(gpu)
    print(
        f"GPU {gpu}: {allocated:.2f} GiB allocated, "
        f"{free / 2**30:.2f} GiB free of {total / 2**30:.2f} GiB"
    )

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at /kaggle/working/b1_llava_med_model were not used when initializing LlavaMistralForCausalLM: ['model.vision_tower.vision_tower.vision_model.encoder.layers.7.mlp.fc2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.12.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.2.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.16.self_attn.q_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.10.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.15.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.16.self_attn.k_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.16.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.8.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.16.layer_norm2.weight', 'model.vision_tower.vision_tow

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Model: microsoft/llava-med-v1.5-mistral-7b
Precision: FP16
Trainable parameters: 0
Vision patches per image: 576
Embeddings: cuda:0
Mistral layer 0: cuda:0
Mistral layer 16: cuda:1
Vision tower: cuda:0
Projector: cuda:0
Output head: cuda:1
GPU 0: 7.60 GiB allocated, 6.82 GiB free of 14.56 GiB
GPU 1: 6.99 GiB allocated, 7.34 GiB free of 14.56 GiB


In [8]:
from collections import defaultdict
from math import sqrt
from safetensors import safe_open
import json

weight_index = json.loads(
    (MODEL_DIR / "model.safetensors.index.json").read_text()
)

prefix = "model.vision_tower.vision_tower."
vision_by_shard = defaultdict(list)

for full_name, shard_name in weight_index["weight_map"].items():
    if full_name.startswith(prefix):
        vision_by_shard[shard_name].append(full_name)

loaded_weights = vision_tower.vision_tower.state_dict()

tensor_count = 0
exact_count = 0
element_count = 0
squared_difference = 0.0
squared_reference = 0.0
largest_difference = (0.0, None)

for shard_name, full_names in vision_by_shard.items():
    with safe_open(
        str(MODEL_DIR / shard_name),
        framework="pt",
        device="cpu",
    ) as checkpoint:
        for full_name in full_names:
            local_name = full_name[len(prefix):]
            assert local_name in loaded_weights, (
                f"Vision weight missing from loaded tower: {local_name}"
            )

            reference = checkpoint.get_tensor(full_name).float()
            loaded = loaded_weights[local_name].detach().cpu().float()
            assert reference.shape == loaded.shape, full_name

            difference = reference - loaded
            maximum = difference.abs().max().item()

            tensor_count += 1
            exact_count += (maximum == 0)
            element_count += reference.numel()
            squared_difference += difference.square().sum().item()
            squared_reference += reference.square().sum().item()

            if maximum > largest_difference[0]:
                largest_difference = (maximum, local_name)

assert tensor_count > 0, "No bundled vision weights found"

print("Bundled vision tensors compared:", tensor_count)
print("Exactly matching tensors:", exact_count)
print("Parameters compared:", element_count)
print(
    "Relative weight difference (RMS):",
    sqrt(squared_difference / squared_reference),
)
print("Largest absolute difference:", largest_difference)

Bundled vision tensors compared: 391
Exactly matching tensors: 0
Parameters compared: 303507456
Relative weight difference (RMS): 0.00166138472505025
Largest absolute difference: (0.046875, 'vision_model.encoder.layers.22.self_attn.k_proj.bias')


In [9]:
from collections import Counter
from safetensors import safe_open

stored_dtypes = Counter()
loaded_dtypes = Counter()
total_elements = 0
matching_after_rounding = 0
largest_remaining_difference = 0.0

for shard_name, full_names in vision_by_shard.items():
    with safe_open(
        str(MODEL_DIR / shard_name),
        framework="pt",
        device="cpu",
    ) as checkpoint:
        for full_name in full_names:
            local_name = full_name[len(prefix):]
            stored = checkpoint.get_tensor(full_name)
            loaded = loaded_weights[local_name].detach().cpu()

            stored_dtypes[str(stored.dtype)] += 1
            loaded_dtypes[str(loaded.dtype)] += 1

            rounded = loaded.to(dtype=stored.dtype)
            total_elements += stored.numel()
            matching_after_rounding += (rounded == stored).sum().item()
            largest_remaining_difference = max(
                largest_remaining_difference,
                (rounded.float() - stored.float()).abs().max().item(),
            )

print("Checkpoint tensor dtypes:", dict(stored_dtypes))
print("Loaded tower tensor dtypes:", dict(loaded_dtypes))
print(
    "Elements matching after rounding:",
    f"{100 * matching_after_rounding / total_elements:.4f}%"
)
print(
    "Largest difference after rounding:",
    largest_remaining_difference
)

Checkpoint tensor dtypes: {'torch.bfloat16': 391}
Loaded tower tensor dtypes: {'torch.float16': 391}
Elements matching after rounding: 99.9855%
Largest difference after rounding: 0.015625


In [10]:
import random
import torch
from PIL import Image

from llava.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IM_END_TOKEN,
)
from llava.conversation import conv_templates
from llava.mm_utils import process_images, tokenizer_image_token

# Use a visual, closed-answer validation question for a clear smoke test.
example = next(
    row for row in english_splits["validation"]
    if row["base_type"] == "vqa"
    and row["answer_type"] == "CLOSED"
)

# The repository's padding function has a possible one-pixel random shift.
# Seed by image ID so every baseline receives the same processed image.
random.seed(int(example["img_id"]))

image_path = image_dir / example["img_name"]
with Image.open(image_path) as image:
    image_tensor = process_images(
        [image], image_processor, model.config
    )[0].unsqueeze(0).to(device="cuda:0", dtype=torch.float16)

question = example["question"].replace(DEFAULT_IMAGE_TOKEN, "").strip()

if model.config.mm_use_im_start_end:
    user_message = (
        DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN +
        DEFAULT_IM_END_TOKEN + "\n" + question
    )
else:
    user_message = DEFAULT_IMAGE_TOKEN + "\n" + question

conversation = conv_templates["mistral_instruct"].copy()
conversation.append_message(conversation.roles[0], user_message)
conversation.append_message(conversation.roles[1], None)
prompt = conversation.get_prompt()

input_ids = tokenizer_image_token(
    prompt,
    tokenizer,
    IMAGE_TOKEN_INDEX,
    return_tensors="pt",
).unsqueeze(0).to("cuda:0")

with torch.inference_mode():
    output_ids = model.generate(
        input_ids,
        images=image_tensor,
        do_sample=False,
        num_beams=1,
        max_new_tokens=64,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

prediction = tokenizer.batch_decode(
    output_ids, skip_special_tokens=True
)[0].strip()

print("Validation question ID:", example["qid"])
print("Image:", example["img_name"])
print("Question:", question)
print("Reference answer:", example["answer"])
print("Model answer:", prediction)
print("Generated tokens:", output_ids.shape[1])

Validation question ID: 9838
Image: xmlab0/source.jpg
Question: Are there abnormalities in this image?
Reference answer: No
Model answer: The abdominal mri image appears to be normal, with no significant abnormalities detected.
Generated tokens: 23


In [11]:
import re
import time
import random
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from PIL import Image
from transformers import LogitsProcessor, LogitsProcessorList

from llava.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IM_END_TOKEN,
)
from llava.conversation import conv_templates
from llava.mm_utils import process_images, tokenizer_image_token


def normalize_answer(value):
    value = unicodedata.normalize("NFKC", str(value)).casefold()
    value = re.sub(r"[^\w\s]", " ", value)
    return " ".join(value.split())


def exact_match(prediction, reference):
    return int(normalize_answer(prediction) == normalize_answer(reference))


class FirstTokenClock(LogitsProcessor):
    def __init__(self):
        self.first_at = None

    def __call__(self, input_ids, scores):
        if self.first_at is None:
            for gpu in (0, 1):
                torch.cuda.synchronize(gpu)
            self.first_at = time.perf_counter()
        return scores


def infer_b1(row, max_new_tokens=64):
    question = row["question"].replace(DEFAULT_IMAGE_TOKEN, "").strip()

    if model.config.mm_use_im_start_end:
        user_message = (
            DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN
            + DEFAULT_IM_END_TOKEN + "\n" + question
        )
    else:
        user_message = DEFAULT_IMAGE_TOKEN + "\n" + question

    conversation = conv_templates["mistral_instruct"].copy()
    conversation.append_message(conversation.roles[0], user_message)
    conversation.append_message(conversation.roles[1], None)

    input_ids = tokenizer_image_token(
        conversation.get_prompt(),
        tokenizer,
        IMAGE_TOKEN_INDEX,
        return_tensors="pt",
    ).unsqueeze(0).to("cuda:0")

    # The repository's image padding uses a random one-pixel offset.
    # Fix it per image so every baseline receives the same image.
    random.seed(str(row["img_name"]).split("/")[0])
    with Image.open(Path(image_dir) / row["img_name"]) as image:
        image_tensor = process_images(
            [image], image_processor, model.config
        ).to(device="cuda:0", dtype=torch.float16)

    for gpu in (0, 1):
        torch.cuda.synchronize(gpu)
        torch.cuda.reset_peak_memory_stats(gpu)

    clock = FirstTokenClock()
    started = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=image_tensor,
            do_sample=False,
            num_beams=1,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            logits_processor=LogitsProcessorList([clock]),
        )
    for gpu in (0, 1):
        torch.cuda.synchronize(gpu)
    finished = time.perf_counter()

    assert clock.first_at is not None, "First-token timing was not recorded"
    prediction = tokenizer.batch_decode(
        output_ids, skip_special_tokens=True
    )[0].strip()

    return {
        "qid": row.get("qid", row.get("id")),
        "img_name": row["img_name"],
        "question": row["question"],
        "reference": row["answer"],
        "answer_type": row["answer_type"],
        "prediction": prediction,
        "exact_match": exact_match(prediction, row["answer"]),
        "returned_token_ids": int(output_ids.shape[-1]),
        "prefill_ms": round(1000 * (clock.first_at - started), 2),
        "total_answer_ms": round(1000 * (finished - started), 2),
        "gpu0_peak_gib": round(torch.cuda.max_memory_allocated(0) / 2**30, 3),
        "gpu1_peak_gib": round(torch.cuda.max_memory_allocated(1) / 2**30, 3),
    }


closed_refs = Counter(
    normalize_answer(row["answer"])
    for row in english_splits["validation"]
    if row["answer_type"] == "CLOSED"
)
nonbinary = sum(
    count for answer, count in closed_refs.items()
    if answer not in {"yes", "no"}
)
print("Validation CLOSED answers other than yes/no:", nonbinary)
print("Most common CLOSED references:", closed_refs.most_common(8))

probe = infer_b1(english_splits["validation"][0])
for key, value in probe.items():
    print(f"{key}: {value}")

Validation CLOSED answers other than yes/no: 64
Most common CLOSED references: [('yes', 182), ('no', 176), ('right', 8), ('liver', 7), ('lung', 6), ('transverse plane', 6), ('heart', 5), ('mri', 4)]
qid: 9835
img_name: xmlab0/source.jpg
question: What modality is used to take this image?
reference: MRI
answer_type: OPEN
prediction: The image is taken using magnetic resonance imaging (MRI), which is a non-invasive imaging technique that uses a strong magnetic field and radio waves to create detailed images of the body's internal structures.
exact_match: 0
returned_token_ids: 49
prefill_ms: 699.46
total_answer_ms: 3765.07
gpu0_peak_gib: 7.773
gpu1_peak_gib: 7.156


In [12]:
import json
import statistics

def lexical_reference_match(prediction, reference):
    pred = normalize_answer(prediction)
    ref = normalize_answer(reference)

    if not ref:
        return None

    if ref in {"yes", "no"}:
        # Prefer an explicit answer at the start of the response.
        leading = re.match(r"^(?:the answer is\s+)?(yes|no)\b", pred)
        if leading:
            return int(leading.group(1) == ref)

        labels = set(re.findall(r"\b(?:yes|no)\b", pred))
        if len(labels) != 1:
            return None  # Neither label, or both labels, appeared.
        return int(next(iter(labels)) == ref)

    # Require a complete word or phrase; "no" must not match "normal".
    found = re.search(rf"(?<!\w){re.escape(ref)}(?!\w)", pred)
    return int(found is not None)


assert lexical_reference_match(
    "The image was acquired using magnetic resonance imaging (MRI).", "MRI"
) == 1
assert lexical_reference_match(
    "No significant abnormalities detected.", "No"
) == 1
assert lexical_reference_match("The scan appears normal.", "No") is None

validation = english_splits["validation"]
groups = [
    ("OPEN / vqa", lambda r: r["answer_type"] == "OPEN"
     and r["base_type"].lower() == "vqa"),
    ("OPEN / kvqa", lambda r: r["answer_type"] == "OPEN"
     and r["base_type"].lower() == "kvqa"),
    ("CLOSED / yes", lambda r: r["answer_type"] == "CLOSED"
     and normalize_answer(r["answer"]) == "yes"),
    ("CLOSED / no", lambda r: r["answer_type"] == "CLOSED"
     and normalize_answer(r["answer"]) == "no"),
    ("CLOSED / other", lambda r: r["answer_type"] == "CLOSED"
     and normalize_answer(r["answer"]) not in {"yes", "no"}),
]

pilot_rows = []
for group_name, select in groups:
    chosen = [row for row in validation if select(row)][:2]
    assert len(chosen) == 2, f"Insufficient validation examples: {group_name}"

    for row in chosen:
        result = infer_b1(row)
        result["group"] = group_name
        result["lexical_reference_match"] = lexical_reference_match(
            result["prediction"], result["reference"]
        )
        pilot_rows.append(result)

        print(f"\n{group_name} | qid={result['qid']}")
        print("Question:", result["question"])
        print("Reference:", result["reference"])
        print("Prediction:", result["prediction"][:300])
        print(
            "Exact:", result["exact_match"],
            "| Lexical check:", result["lexical_reference_match"],
            "| Total ms:", result["total_answer_ms"],
        )

pilot_path = Path("/kaggle/working/b1_slake_validation_pilot.json")
pilot_path.write_text(json.dumps(pilot_rows, indent=2), encoding="utf-8")

print("\n--- Pilot summary ---")
print("Examples:", len(pilot_rows))
print("Exact matches:", sum(r["exact_match"] for r in pilot_rows))
print(
    "Lexical hits / resolved:",
    sum(r["lexical_reference_match"] == 1 for r in pilot_rows),
    "/",
    sum(r["lexical_reference_match"] is not None for r in pilot_rows),
)
print("Median prefill ms:", statistics.median(r["prefill_ms"] for r in pilot_rows))
print("Median total ms:", statistics.median(r["total_answer_ms"] for r in pilot_rows))
print("Max GPU0 peak GiB:", max(r["gpu0_peak_gib"] for r in pilot_rows))
print("Max GPU1 peak GiB:", max(r["gpu1_peak_gib"] for r in pilot_rows))
print("Saved:", pilot_path)


OPEN / vqa | qid=9835
Question: What modality is used to take this image?
Reference: MRI
Prediction: The image is taken using magnetic resonance imaging (MRI), which is a non-invasive imaging technique that uses a strong magnetic field and radio waves to create detailed images of the body's internal structures.
Exact: 0 | Lexical check: 1 | Total ms: 3776.9

OPEN / vqa | qid=9836
Question: Which part of the body does this image belong to?
Reference: Abdomen
Prediction: The image belongs to the abdomen, specifically focusing on the pancreas.
Exact: 0 | Lexical check: 1 | Total ms: 1892.59

OPEN / kvqa | qid=10153
Question: How much damage will be caused to the body taking such an image?
Reference: A Little
Prediction: The chest X-ray image is considered to be of low radiation dose. This means that the amount of radiation exposure is relatively low compared to other types of medical imaging. However, it is important to note that any exposure to ionizing radiation, such as that from an X

In [13]:
SHORT_ANSWER_INSTRUCTION = (
    "Answer using only the shortest appropriate word or phrase. "
    "For a yes/no question, answer exactly Yes or No. "
    "Do not explain, repeat the question, or list choices."
)

def infer_b1_short(row):
    prompted_row = dict(row)
    prompted_row["question"] = (
        row["question"].strip() + "\n" + SHORT_ANSWER_INSTRUCTION
    )

    result = infer_b1(prompted_row, max_new_tokens=64)
    result["question"] = row["question"]  # Preserve the original question.
    result["prompt_instruction"] = SHORT_ANSWER_INSTRUCTION
    return result


validation_by_qid = {
    str(row.get("qid", row.get("id"))): row
    for row in english_splits["validation"]
}

short_pilot = []
for old in pilot_rows:
    row = validation_by_qid[str(old["qid"])]
    new = infer_b1_short(row)
    new["group"] = old["group"]
    short_pilot.append(new)

    print(f"\n{old['group']} | qid={old['qid']}")
    print("Reference:", old["reference"])
    print("Original:", old["prediction"][:350])
    print("Short prompt:", new["prediction"][:350])
    print(
        "Exact match: original =", old["exact_match"],
        "| short prompt =", new["exact_match"],
        "| new total ms =", new["total_answer_ms"],
    )

short_path = Path("/kaggle/working/b1_slake_short_prompt_validation_pilot.json")
short_path.write_text(json.dumps(short_pilot, indent=2), encoding="utf-8")

print("\n--- Comparison on the same 10 validation questions ---")
print("Original exact matches:", sum(r["exact_match"] for r in pilot_rows))
print("Short-prompt exact matches:", sum(r["exact_match"] for r in short_pilot))
print(
    "Original median answer ms:",
    statistics.median(r["total_answer_ms"] for r in pilot_rows),
)
print(
    "Short-prompt median answer ms:",
    statistics.median(r["total_answer_ms"] for r in short_pilot),
)
print(
    "Short-prompt outputs reaching 64-token limit:",
    sum(r["returned_token_ids"] >= 64 for r in short_pilot),
)
print("Saved:", short_path)


OPEN / vqa | qid=9835
Reference: MRI
Original: The image is taken using magnetic resonance imaging (MRI), which is a non-invasive imaging technique that uses a strong magnetic field and radio waves to create detailed images of the body's internal structures.
Short prompt: The image is taken using magnetic resonance imaging (MRI).
Exact match: original = 0 | short prompt = 0 | new total ms = 1705.08

OPEN / vqa | qid=9836
Reference: Abdomen
Original: The image belongs to the abdomen, specifically focusing on the pancreas.
Short prompt: The image belongs to the abdomen.
Exact match: original = 0 | short prompt = 0 | new total ms = 1294.86

OPEN / kvqa | qid=10153
Reference: A Little
Original: The chest X-ray image is considered to be of low radiation dose. This means that the amount of radiation exposure is relatively low compared to other types of medical imaging. However, it is important to note that any exposure to ionizing radiation, such as that from an X-ray, carries some risk. Th

In [14]:
import json
import math
import statistics
import subprocess
from pathlib import Path

import transformers
from tqdm.auto import tqdm

test_rows = list(english_splits["test"])
assert len(test_rows) == 1061

output_path = Path("/kaggle/working/b1_slake_test_raw.jsonl")
protocol_path = Path("/kaggle/working/b1_slake_test_protocol.json")
summary_path = Path("/kaggle/working/b1_slake_test_summary.json")

def question_id(row):
    return str(row.get("qid", row.get("id")))

assert len({question_id(row) for row in test_rows}) == len(test_rows)

repo_sha = subprocess.check_output(
    ["git", "-C", "/kaggle/working/LLaVA-Med", "rev-parse", "HEAD"],
    text=True,
).strip()

protocol = {
    "baseline": "B1: LLaVA-Med v1.5, no visual-token pruning",
    "dataset": "SLAKE 1.0, English test split",
    "dataset_revision": "a9083ce6c34ac3ffb17671a605962924d8a8f9e9",
    "test_questions": len(test_rows),
    "model": "microsoft/llava-med-v1.5-mistral-7b",
    "model_revision": "91bb16c122001ddc9cf1fd36ce1dae09448943a2",
    "llava_repo_revision": repo_sha,
    "conversation_template": "mistral_instruct",
    "question_prompt": "original question, no added answer instruction",
    "decoding": {
        "do_sample": False,
        "num_beams": 1,
        "max_new_tokens": 1024,
        "use_cache": True,
    },
    "precision": "float16",
    "gpu_layout": "model sharded across two Tesla T4 GPUs",
    "visual_tokens": 576,
    "visual_token_retention": 1.0,
    "prefill_definition": (
        "GPU-synchronized time from generation start through "
        "the first logits-processor invocation; includes vision encoding"
    ),
    "total_time_definition": (
        "GPU-synchronized time for model.generate; excludes image loading "
        "and preprocessing"
    ),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
}

if protocol_path.exists():
    saved_protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
    assert saved_protocol == protocol, (
        "An existing output uses a different protocol. "
        "Inspect its files before resuming."
    )
else:
    assert not output_path.exists() or output_path.stat().st_size == 0, (
        "Predictions exist without a protocol file; inspect before resuming."
    )
    protocol_path.write_text(
        json.dumps(protocol, indent=2), encoding="utf-8"
    )

completed = []
if output_path.exists():
    with output_path.open("r", encoding="utf-8") as file:
        completed = [json.loads(line) for line in file if line.strip()]

assert len(completed) <= len(test_rows)
for old, row in zip(completed, test_rows):
    assert str(old["qid"]) == question_id(row), (
        "Saved predictions do not match the current test order."
    )

print(f"Already completed: {len(completed)}/{len(test_rows)}")
print("Raw predictions:", output_path)

records = list(completed)
with output_path.open("a", encoding="utf-8", buffering=1) as file:
    with tqdm(
        total=len(test_rows),
        initial=len(completed),
        desc="B1 SLAKE English test",
    ) as progress:
        for row in test_rows[len(completed):]:
            result = infer_b1(row, max_new_tokens=1024)
            result["base_type"] = row["base_type"]
            result["split"] = "test"

            file.write(json.dumps(result, ensure_ascii=False) + "\n")
            file.flush()
            records.append(result)
            progress.update(1)

            if len(records) % 25 == 0:
                progress.set_postfix(
                    latest_ms=result["total_answer_ms"],
                    gpu0_gib=result["gpu0_peak_gib"],
                    gpu1_gib=result["gpu1_peak_gib"],
                )

assert len(records) == len(test_rows)

def timing_summary(key):
    values = sorted(record[key] for record in records)
    return {
        "mean_ms": round(statistics.mean(values), 2),
        "median_ms": round(statistics.median(values), 2),
        "p95_ms": round(values[math.ceil(0.95 * len(values)) - 1], 2),
    }

def strict_exact_summary(items):
    return {
        "correct": sum(item["exact_match"] for item in items),
        "total": len(items),
        "rate": round(
            sum(item["exact_match"] for item in items) / len(items), 4
        ),
    }

summary = {
    "protocol": protocol,
    "strict_exact_match_not_semantic_accuracy": {
        "overall": strict_exact_summary(records),
        "OPEN": strict_exact_summary(
            [r for r in records if r["answer_type"] == "OPEN"]
        ),
        "CLOSED": strict_exact_summary(
            [r for r in records if r["answer_type"] == "CLOSED"]
        ),
    },
    "prefill": timing_summary("prefill_ms"),
    "total_answer": timing_summary("total_answer_ms"),
    "peak_allocated_gib": {
        "gpu0_max": max(r["gpu0_peak_gib"] for r in records),
        "gpu1_max": max(r["gpu1_peak_gib"] for r in records),
    },
    "outputs_at_token_cap": sum(
        r["returned_token_ids"] >= 1024 for r in records
    ),
}

summary_path.write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

print("\nCompleted:", len(records))
print("Strict exact match:", summary["strict_exact_match_not_semantic_accuracy"])
print("Prefill timing:", summary["prefill"])
print("Total-answer timing:", summary["total_answer"])
print("Peak allocation:", summary["peak_allocated_gib"])
print("Outputs at token cap:", summary["outputs_at_token_cap"])
print("Saved:", output_path, protocol_path, summary_path, sep="\n")

Already completed: 0/1061
Raw predictions: /kaggle/working/b1_slake_test_raw.jsonl


B1 SLAKE English test:   0%|          | 0/1061 [00:00<?, ?it/s]


Completed: 1061
Strict exact match: {'overall': {'correct': 0, 'total': 1061, 'rate': 0.0}, 'OPEN': {'correct': 0, 'total': 645, 'rate': 0.0}, 'CLOSED': {'correct': 0, 'total': 416, 'rate': 0.0}}
Prefill timing: {'mean_ms': 610.67, 'median_ms': 612.63, 'p95_ms': 636.7}
Total-answer timing: {'mean_ms': 2321.0, 'median_ms': 1841.12, 'p95_ms': 4800.4}
Peak allocation: {'gpu0_max': 7.843, 'gpu1_max': 7.252}
Outputs at token cap: 1
Saved:
/kaggle/working/b1_slake_test_raw.jsonl
/kaggle/working/b1_slake_test_protocol.json
/kaggle/working/b1_slake_test_summary.json


In [15]:
import json
import re
import statistics
from collections import Counter
from pathlib import Path

raw_path = Path("/kaggle/working/b1_slake_test_raw.jsonl")
test_predictions = [
    json.loads(line)
    for line in raw_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert len(test_predictions) == 1061
assert [str(r["qid"]) for r in test_predictions] == [
    str(row.get("qid", row.get("id")))
    for row in english_splits["test"]
]
assert all(
    r["exact_match"] == exact_match(r["prediction"], r["reference"])
    for r in test_predictions
)

def leading_binary_answer(prediction):
    text = normalize_answer(prediction)
    match = re.match(
        r"^(?:(?:the )?answer is\s+)?(yes|no)\b",
        text,
    )
    return match.group(1) if match else None


binary = [
    r for r in test_predictions
    if normalize_answer(r["reference"]) in {"yes", "no"}
]
explicit = [
    (r, leading_binary_answer(r["prediction"]))
    for r in binary
    if leading_binary_answer(r["prediction"]) is not None
]
correct_explicit = sum(
    label == normalize_answer(r["reference"])
    for r, label in explicit
)

word_counts = sorted(
    len(normalize_answer(r["prediction"]).split())
    for r in test_predictions
)

print("Test rows:", len(test_predictions))
print("Strict exact matches:", sum(r["exact_match"] for r in test_predictions))
print(
    "Prediction length in words: min / median / max:",
    word_counts[0],
    statistics.median(word_counts),
    word_counts[-1],
)
print("Binary-reference questions:", len(binary))
print(
    "Explicit leading Yes/No:",
    len(explicit),
    f"({len(explicit) / len(binary):.1%} coverage)",
)
print(
    "Correct among explicit leading Yes/No:",
    correct_explicit,
    "/",
    len(explicit),
    (
        f"({correct_explicit / len(explicit):.1%})"
        if explicit else "(none)"
    ),
)
print(
    "Explicit Yes/No confusion counts:",
    dict(Counter(
        (normalize_answer(r["reference"]), label)
        for r, label in explicit
    )),
)

print("\nShortest eight model responses:")
for r in sorted(
    test_predictions,
    key=lambda x: len(normalize_answer(x["prediction"]).split()),
)[:8]:
    print(
        f"qid={r['qid']} | reference={r['reference']!r} "
        f"| prediction={r['prediction']!r}"
    )

print("\nFive binary responses without a leading Yes/No:")
indirect = [
    r for r in binary
    if leading_binary_answer(r["prediction"]) is None
]
for r in indirect[:5]:
    print(
        f"qid={r['qid']} | reference={r['reference']!r} "
        f"| prediction={r['prediction'][:220]!r}"
    )

Test rows: 1061
Strict exact matches: 0
Prediction length in words: min / median / max: 7 13 286
Binary-reference questions: 355
Explicit leading Yes/No: 316 (89.0% coverage)
Correct among explicit leading Yes/No: 198 / 316 (62.7%)
Explicit Yes/No confusion counts: {('no', 'no'): 74, ('yes', 'yes'): 124, ('no', 'yes'): 96, ('yes', 'no'): 22}

Shortest eight model responses:
qid=12058 | reference='1' | prediction='There are two kidneys in this image.'
qid=12061 | reference='Yes' | prediction='Yes, the MRI image shows the kidney.'
qid=12115 | reference='0' | prediction='In this image, there are two kidneys.'
qid=12146 | reference='2' | prediction='In this image, there are two kidneys.'
qid=12157 | reference='0' | prediction='There are two kidneys in this image.'
qid=12168 | reference='0' | prediction='In this image, there are two kidneys.'
qid=12229 | reference='2' | prediction='There are two kidneys in this image.'
qid=12257 | reference='0' | prediction='In this image, there are two kid

In [16]:
import json
import re
from collections import defaultdict
from pathlib import Path

indirect = [
    r for r in binary
    if leading_binary_answer(r["prediction"]) is None
]
assert len(indirect) == len(binary) - len(explicit)

print("Binary leading-label agreement:", correct_explicit, "/", len(explicit))
print(
    "Binary agreement bounds before reviewing indirect answers:",
    f"{correct_explicit / len(binary):.1%}",
    "to",
    f"{(correct_explicit + len(indirect)) / len(binary):.1%}",
)
print("Indirect binary responses requiring review:", len(indirect))


def contains_reference(text, reference):
    text = normalize_answer(text)
    reference = normalize_answer(reference)
    if not reference:
        return False
    return re.search(
        rf"(?<!\w){re.escape(reference)}(?!\w)", text
    ) is not None


for label, subset in [
    (
        "OPEN",
        [r for r in test_predictions if r["answer_type"] == "OPEN"],
    ),
    (
        "CLOSED with nonbinary reference",
        [
            r for r in test_predictions
            if r["answer_type"] == "CLOSED"
            and normalize_answer(r["reference"]) not in {"yes", "no"}
        ],
    ),
]:
    mentioned = [
        r for r in subset
        if contains_reference(r["prediction"], r["reference"])
    ]
    echoed = [
        r for r in mentioned
        if contains_reference(r["question"], r["reference"])
    ]
    print(f"\n{label}: {len(subset)} questions")
    print("Reference phrase appears in prediction:", len(mentioned))
    print(
        "Of those, reference phrase also appears in question:",
        len(echoed),
    )

groups = defaultdict(list)
for r in indirect:
    key = (
        normalize_answer(r["question"]),
        normalize_answer(r["reference"]),
        normalize_answer(r["prediction"]),
    )
    groups[key].append(r)

print("\n--- Indirect binary responses, grouped ---")
print("Distinct question/reference/prediction groups:", len(groups))
for group in sorted(groups.values(), key=len, reverse=True):
    first = group[0]
    ids = ", ".join(str(r["qid"]) for r in group)
    print(f"\nCount={len(group)} | qids={ids}")
    print("Question:", first["question"])
    print("Reference:", first["reference"])
    print("Prediction:", first["prediction"][:500])

review_path = Path("/kaggle/working/b1_binary_indirect_review.json")
review_path.write_text(
    json.dumps(indirect, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("\nSaved review file:", review_path)

Binary leading-label agreement: 198 / 316
Binary agreement bounds before reviewing indirect answers: 55.8% to 66.8%
Indirect binary responses requiring review: 39

OPEN: 645 questions
Reference phrase appears in prediction: 202
Of those, reference phrase also appears in question: 0

CLOSED with nonbinary reference: 61 questions
Reference phrase appears in prediction: 40
Of those, reference phrase also appears in question: 40

--- Indirect binary responses, grouped ---
Distinct question/reference/prediction groups: 16

Count=10 | qids=11970, 11989, 11996, 12003, 12011, 12053, 12073, 12421, 12451, 12467
Question: Are there abnormalities in this image?
Reference: Yes
Prediction: The chest x-ray appears to be normal, with no significant abnormalities detected.

Count=4 | qids=12132, 12147, 12172, 12192
Question: Does this image look normal?
Reference: No
Prediction: The image appears to be normal, as it shows no abnormalities.

Count=4 | qids=12332, 12361, 12466, 12512
Question: Does the l

In [17]:
import csv
import json
import re
from pathlib import Path

binary_judgments = []

for row in binary:
    direct_label = leading_binary_answer(row["prediction"])

    if direct_label is not None:
        predicted_label = direct_label
        method = "explicit_leading_yes_no"
    else:
        prediction = normalize_answer(row["prediction"])
        question = normalize_answer(row["question"])

        # We reviewed all 39 indirect outputs from Cell 14.
        # Each describes normal appearance or absence of the condition.
        assert re.search(
            r"\bnormal\b|\bno\b|\bdoes not have\b", prediction
        ), f"Unreviewed indirect wording in qid={row['qid']}"

        # A normal image answers "Is it normal?" with Yes;
        # it answers "Is it abnormal / does it have X?" with No.
        predicted_label = (
            "yes" if re.search(r"\bnormal\b", question) else "no"
        )
        method = "reviewed_normal_or_absence_statement"

    reference_label = normalize_answer(row["reference"])
    binary_judgments.append({
        "qid": row["qid"],
        "question": row["question"],
        "reference": row["reference"],
        "prediction": row["prediction"],
        "predicted_label": predicted_label,
        "reference_label": reference_label,
        "agreement": int(predicted_label == reference_label),
        "method": method,
    })

binary_correct = sum(r["agreement"] for r in binary_judgments)
indirect_correct = sum(
    r["agreement"] for r in binary_judgments
    if r["method"] == "reviewed_normal_or_absence_statement"
)

print("Binary reference questions:", len(binary_judgments))
print("Correct leading Yes/No:", correct_explicit, "/", len(explicit))
print("Correct reviewed indirect:", indirect_correct, "/", len(indirect))
print(
    "Combined binary answer agreement:",
    binary_correct, "/", len(binary_judgments),
    f"= {binary_correct / len(binary_judgments):.1%}",
)

binary_path = Path("/kaggle/working/b1_slake_binary_judgments.json")
binary_path.write_text(
    json.dumps(binary_judgments, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

nonbinary = [
    row for row in test_predictions
    if normalize_answer(row["reference"]) not in {"yes", "no"}
]
assert len(nonbinary) == 706

review_path = Path("/kaggle/working/b1_slake_nonbinary_review.csv")
fields = [
    "qid", "answer_type", "base_type", "question",
    "reference", "prediction", "semantic_equivalence", "review_notes",
]
with review_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    for row in nonbinary:
        writer.writerow({
            "qid": row["qid"],
            "answer_type": row["answer_type"],
            "base_type": row["base_type"],
            "question": row["question"],
            "reference": row["reference"],
            "prediction": row["prediction"],
            "semantic_equivalence": "",
            "review_notes": "",
        })

print("Nonbinary answers awaiting review:", len(nonbinary))
print("Saved:", binary_path)
print("Saved:", review_path)

Binary reference questions: 355
Correct leading Yes/No: 198 / 316
Correct reviewed indirect: 3 / 39
Combined binary answer agreement: 201 / 355 = 56.6%
Nonbinary answers awaiting review: 706
Saved: /kaggle/working/b1_slake_binary_judgments.json
Saved: /kaggle/working/b1_slake_nonbinary_review.csv


In [18]:
import base64
import hashlib
import json
import re
import zlib
from collections import Counter
from pathlib import Path

ROOT = Path("/kaggle/working")
RAW = ROOT / "b1_slake_test_raw.jsonl"
assert RAW.is_file(), f"Missing inference output: {RAW}"

# Reviewed nonbinary labels in ascending question-ID order:
# 1 = matching answer, 0 = mismatch, U = uncertain.
LABELS_ZLIB_B64 = (
    "eJxlUokNxDAMWgmGYf9VWgNOejpVSvPYfAkpQCCR3zuRF2QW2NWO/NtJTStnlJHuIQtpjkBrVp6YP3RKU3BwOwbG5cAhoSolyG3KpFvz6f3GkZSGUygeuW7RHg+XgGPOlhVVqmxEOCNtLKrHblPCxOZZAKHnznXZPvr1FWjVrVIFvQwJNki72Zmavo074XtRyr2MURxE1lju3PLU6Db9wvaBbD1LG2bxw6SfbovQvbqNog8pjj1tds57nNeltvyOTTA1wAP5kZLE"
)
NONBINARY_QIDS_SHA256 = "caee30cbed278617599a5f23c39e27692e42f520f37d1aa4f2bb956c43dc6201"
NONBINARY_PREDICTIONS_SHA256 = "1bca305300f5e55404ff4e8717367cd4344d57964169e35e0749e62f2e5dbbe2"
INDIRECT_BINARY_CORRECT = {12092, 12209, 12771}

def field(row, *names):
    for name in names:
        if name in row and row[name] is not None:
            return str(row[name])
    raise KeyError(f"Expected one of {names}; available keys: {list(row)}")

with RAW.open(encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
assert len(rows) == 1061
by_qid = {int(field(r, "qid", "question_id")): r for r in rows}
assert len(by_qid) == 1061, "Duplicate test qid"

def reference(row):
    return field(row, "reference", "gt_answer", "reference_answer", "answer")

def prediction(row):
    return field(row, "prediction", "model_answer", "raw_prediction")

binary_ids = sorted(
    qid for qid, r in by_qid.items()
    if reference(r).strip().lower() in {"yes", "no"}
)
nonbinary_ids = sorted(set(by_qid) - set(binary_ids))
assert (len(binary_ids), len(nonbinary_ids)) == (355, 706)

qid_hash = hashlib.sha256(",".join(map(str, nonbinary_ids)).encode()).hexdigest()
assert qid_hash == NONBINARY_QIDS_SHA256, "Review qids differ from this test output"
prediction_hash = hashlib.sha256(
    "".join(
        f"{qid}\t{prediction(by_qid[qid])}\n"
        for qid in nonbinary_ids
    ).encode("utf-8")
).hexdigest()
assert prediction_hash == NONBINARY_PREDICTIONS_SHA256, (
    "Raw predictions differ from the reviewed CSV; stop and inspect"
)

labels = zlib.decompress(base64.b64decode(LABELS_ZLIB_B64)).decode("ascii")
assert len(labels) == 706 and set(labels) == {"0", "1", "U"}
nonbinary = dict(zip(nonbinary_ids, labels))

leading = 0
leading_correct = 0
indirect_ids = set()
judgments = []

for qid in sorted(by_qid):
    row = by_qid[qid]
    ref = reference(row).strip()
    pred = prediction(row)
    answer_type = field(row, "answer_type").upper()
    base_type = field(row, "base_type").lower()

    if qid in nonbinary:
        label = nonbinary[qid]
        review = "text_review"
    else:
        assert answer_type == "CLOSED"
        match = re.match(r"\s*(yes|no)\b", pred, re.I)
        if match:
            leading += 1
            label = "1" if match.group(1).lower() == ref.lower() else "0"
            leading_correct += label == "1"
            review = "explicit_yes_no"
        else:
            indirect_ids.add(qid)
            label = "1" if qid in INDIRECT_BINARY_CORRECT else "0"
            review = "manual_indirect_binary"

    judgments.append({
        "qid": qid,
        "answer_type": answer_type,
        "base_type": base_type,
        "reference": ref,
        "prediction": pred,
        "semantic_equivalence": label,
        "review_source": review,
    })

assert (leading, leading_correct, len(indirect_ids)) == (316, 198, 39)
assert INDIRECT_BINARY_CORRECT <= indirect_ids
assert Counter(j["answer_type"] for j in judgments) == {
    "OPEN": 645, "CLOSED": 416
}
assert Counter(
    j["semantic_equivalence"]
    for j in judgments if j["qid"] in nonbinary
) == {"1": 279, "0": 335, "U": 92}

def summarize(subset):
    n = len(subset)
    c = Counter(j["semantic_equivalence"] for j in subset)
    return {
        "correct": c["1"],
        "incorrect": c["0"],
        "uncertain": c["U"],
        "total": n,
        "lower_rate": round(c["1"] / n, 4),
        "upper_rate": round((c["1"] + c["U"]) / n, 4),
        "resolved_only_rate": round(c["1"] / (n - c["U"]), 4),
    }

summary = {
    "metric": "text-only semantic answer agreement with SLAKE references",
    "labels": "1=matching answer; 0=wrong or no answer; U=ambiguous or partial",
    "overall": summarize(judgments),
    "OPEN": summarize([j for j in judgments if j["answer_type"] == "OPEN"]),
    "CLOSED": summarize([j for j in judgments if j["answer_type"] == "CLOSED"]),
    "binary_reference": summarize(
        [j for j in judgments if j["qid"] in binary_ids]
    ),
    "nonbinary_reference": summarize(
        [j for j in judgments if j["qid"] in nonbinary]
    ),
    "leading_binary": {"correct": leading_correct, "total": leading},
    "indirect_binary": {
        "correct": len(INDIRECT_BINARY_CORRECT),
        "total": len(indirect_ids),
    },
    "reviewed_nonbinary_prediction_sha256": prediction_hash,
}

judgments_path = ROOT / "b1_slake_test_semantic_judgments.jsonl"
summary_path = ROOT / "b1_slake_test_semantic_summary.json"

with judgments_path.open("w", encoding="utf-8") as f:
    for judgment in judgments:
        f.write(json.dumps(judgment, ensure_ascii=False) + "\n")
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

print(json.dumps(summary, indent=2))
print("Saved:", judgments_path, summary_path, sep="\n")

{
  "metric": "text-only semantic answer agreement with SLAKE references",
  "labels": "1=matching answer; 0=wrong or no answer; U=ambiguous or partial",
  "overall": {
    "correct": 480,
    "incorrect": 489,
    "uncertain": 92,
    "total": 1061,
    "lower_rate": 0.4524,
    "upper_rate": 0.5391,
    "resolved_only_rate": 0.4954
  },
  "OPEN": {
    "correct": 249,
    "incorrect": 305,
    "uncertain": 91,
    "total": 645,
    "lower_rate": 0.386,
    "upper_rate": 0.5271,
    "resolved_only_rate": 0.4495
  },
  "CLOSED": {
    "correct": 231,
    "incorrect": 184,
    "uncertain": 1,
    "total": 416,
    "lower_rate": 0.5553,
    "upper_rate": 0.5577,
    "resolved_only_rate": 0.5566
  },
  "binary_reference": {
    "correct": 201,
    "incorrect": 154,
    "uncertain": 0,
    "total": 355,
    "lower_rate": 0.5662,
    "upper_rate": 0.5662,
    "resolved_only_rate": 0.5662
  },
  "nonbinary_reference": {
    "correct": 279,
    "incorrect": 335,
    "uncertain": 92,
    "tota

In [19]:
import hashlib
import json
import zipfile
from pathlib import Path

root = Path("/kaggle/working")
files = [
    "b1_slake_test_raw.jsonl",
    "b1_slake_test_protocol.json",
    "b1_slake_test_summary.json",
    "b1_slake_test_semantic_judgments.jsonl",
    "b1_slake_test_semantic_summary.json",
]
for name in files:
    assert (root / name).is_file(), f"Missing: {name}"

semantic = json.loads(
    (root / "b1_slake_test_semantic_summary.json").read_text()
)
assert semantic["overall"]["total"] == 1061
assert (
    semantic["overall"]["correct"],
    semantic["overall"]["incorrect"],
    semantic["overall"]["uncertain"],
) == (480, 489, 92)

manifest = {
    "baseline": "B1",
    "configuration": "LLaVA-Med; no pruning; 100% visual tokens retained",
    "backbone_weights": "frozen",
    "model": "microsoft/llava-med-v1.5-mistral-7b",
    "model_revision": "91bb16c122001ddc9cf1fd36ce1dae09448943a2",
    "dataset": "BoKelvin/SLAKE",
    "dataset_revision": "a9083ce6c34ac3ffb17671a605962924d8a8f9e9",
    "evaluation_split": "English test",
    "test_questions": 1061,
    "semantic_scoring": semantic,
    "timing_ms": {
        "prefill": {"mean": 610.67, "median": 612.63, "p95": 636.70},
        "total_answer": {"mean": 2321.00, "median": 1841.12, "p95": 4800.40},
    },
    "peak_pytorch_allocation_gib": {"gpu0": 7.843, "gpu1": 7.252},
    "protocol_file": "b1_slake_test_protocol.json",
    "inference_summary_file": "b1_slake_test_summary.json",
    "note": (
        "Semantic judgments assess prediction text against references. "
        "The 92 uncertain answers remain unresolved; the lower and upper "
        "rates are bounds, not an official single-number accuracy."
    ),
    "file_sha256": {
        name: hashlib.sha256((root / name).read_bytes()).hexdigest()
        for name in files
    },
}

manifest_path = root / "b1_slake_baseline_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

bundle_path = root / "b1_slake_baseline_bundle.zip"
with zipfile.ZipFile(
    bundle_path, "w", compression=zipfile.ZIP_DEFLATED
) as bundle:
    for name in files:
        bundle.write(root / name, arcname=name)
    bundle.write(manifest_path, arcname=manifest_path.name)

print("B1 packaged:", bundle_path)
print("Files:", len(files) + 1)
print("Clear matches:", semantic["overall"]["correct"], "/ 1061")
print("Uncertain:", semantic["overall"]["uncertain"])
print("Bundle SHA256:", hashlib.sha256(bundle_path.read_bytes()).hexdigest())

B1 packaged: /kaggle/working/b1_slake_baseline_bundle.zip
Files: 6
Clear matches: 480 / 1061
Uncertain: 92
Bundle SHA256: b9ae0e460186b371042e1cd6998499b13626e8e1e57c28347fc800064e25fe49


In [20]:
import hashlib
import json
import zipfile
from pathlib import Path

root = Path("/kaggle/working")
files = [
    "b1_slake_test_raw.jsonl",
    "b1_slake_test_protocol.json",
    "b1_slake_test_summary.json",
    "b1_slake_test_semantic_judgments.jsonl",
    "b1_slake_test_semantic_summary.json",
]
for name in files:
    assert (root / name).is_file(), f"Missing: {name}"

semantic = json.loads(
    (root / "b1_slake_test_semantic_summary.json").read_text()
)
assert semantic["overall"]["total"] == 1061
assert (
    semantic["overall"]["correct"],
    semantic["overall"]["incorrect"],
    semantic["overall"]["uncertain"],
) == (480, 489, 92)

manifest = {
    "baseline": "B1",
    "configuration": "LLaVA-Med; no pruning; 100% visual tokens retained",
    "backbone_weights": "frozen",
    "model": "microsoft/llava-med-v1.5-mistral-7b",
    "model_revision": "91bb16c122001ddc9cf1fd36ce1dae09448943a2",
    "dataset": "BoKelvin/SLAKE",
    "dataset_revision": "a9083ce6c34ac3ffb17671a605962924d8a8f9e9",
    "evaluation_split": "English test",
    "test_questions": 1061,
    "semantic_scoring": semantic,
    "timing_ms": {
        "prefill": {"mean": 610.67, "median": 612.63, "p95": 636.70},
        "total_answer": {"mean": 2321.00, "median": 1841.12, "p95": 4800.40},
    },
    "peak_pytorch_allocation_gib": {"gpu0": 7.843, "gpu1": 7.252},
    "protocol_file": "b1_slake_test_protocol.json",
    "inference_summary_file": "b1_slake_test_summary.json",
    "note": (
        "Semantic judgments assess prediction text against references. "
        "The 92 uncertain answers remain unresolved; the lower and upper "
        "rates are bounds, not an official single-number accuracy."
    ),
    "file_sha256": {
        name: hashlib.sha256((root / name).read_bytes()).hexdigest()
        for name in files
    },
}

manifest_path = root / "b1_slake_baseline_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

bundle_path = root / "b1_slake_baseline_bundle.zip"
with zipfile.ZipFile(
    bundle_path, "w", compression=zipfile.ZIP_DEFLATED
) as bundle:
    for name in files:
        bundle.write(root / name, arcname=name)
    bundle.write(manifest_path, arcname=manifest_path.name)

print("B1 packaged:", bundle_path)
print("Files:", len(files) + 1)
print("Clear matches:", semantic["overall"]["correct"], "/ 1061")
print("Uncertain:", semantic["overall"]["uncertain"])
print("Bundle SHA256:", hashlib.sha256(bundle_path.read_bytes()).hexdigest())

B1 packaged: /kaggle/working/b1_slake_baseline_bundle.zip
Files: 6
Clear matches: 480 / 1061
Uncertain: 92
Bundle SHA256: 95cc0e4c95841e0e7bf6f5719138920a2fb4eb356a8a68e15632ebf6cd769132
